# LegalBenchRAG — Sample 50 Queries per Dataset

Deterministically samples 50 queries from each of the four LegalBenchRAG benchmark files
(`contractnli`, `cuad`, `maud`, `privacy_qa`) and saves the combined result to a CSV.

**Seed**: `42` — rerunning this notebook always produces the same sample.

In [1]:
import json
import random
import pathlib
import pandas as pd

In [2]:
BENCHMARKS_DIR = pathlib.Path("../data/LegalBenchRAG/benchmarks")
OUTPUT_DIR     = pathlib.Path("../data/LegalBenchRAG/benchmarks_50")
OUTPUT_CSV     = OUTPUT_DIR.parent / "sampled_queries.csv"

SAMPLE_SIZE = 50
SEED        = 42

DATASETS = ["contractnli", "cuad", "maud", "privacy_qa"]

In [3]:
def load_tests(name: str) -> list[dict]:
    path = BENCHMARKS_DIR / f"{name}.json"
    with open(path) as f:
        return json.load(f)["tests"]


def flatten_test(test: dict, dataset: str) -> dict:
    """Flatten a single test entry into a row-friendly dict.

    Each snippet becomes a separate column group (snippet_0_*, snippet_1_*, …).
    For simpler downstream use, the first snippet's fields are also promoted to
    top-level columns (file_path, span_start, span_end, answer).
    """
    row = {"dataset": dataset, "query": test["query"]}

    snippets = test.get("snippets", [])
    row["num_snippets"] = len(snippets)

    # Primary snippet (index 0) promoted to top-level for easy filtering
    if snippets:
        row["file_path"]  = snippets[0].get("file_path", "")
        span = snippets[0].get("span", [None, None])
        row["span_start"] = span[0] if len(span) > 0 else None
        row["span_end"]   = span[1] if len(span) > 1 else None
        row["answer"]     = snippets[0].get("answer", "")
    else:
        row.update({"file_path": "", "span_start": None, "span_end": None, "answer": ""})

    # All snippets stored as JSON string for completeness
    row["snippets_json"] = json.dumps(snippets)

    return row

In [4]:
rng = random.Random(SEED)

sampled_by_dataset: dict[str, list[dict]] = {}
rows = []

for dataset in DATASETS:
    tests = load_tests(dataset)
    n_available = len(tests)
    n_sample    = min(SAMPLE_SIZE, n_available)
    sampled     = rng.sample(tests, n_sample)
    sampled_by_dataset[dataset] = sampled
    print(f"{dataset:>15s}: {n_available:5d} total → sampled {n_sample}")
    for test in sampled:
        rows.append(flatten_test(test, dataset))

df = pd.DataFrame(rows)
print(f"\nTotal rows: {len(df)}")
df.head()

    contractnli:   977 total → sampled 50
           cuad:  4042 total → sampled 50
           maud:  1676 total → sampled 50
     privacy_qa:   194 total → sampled 50

Total rows: 200


,dataset,query,num_snippets,file_path,span_start,span_end,answer,snippets_json
0,contractnli,Consider the Tabun Kitchen Investments' Non-Di...,2,contractnli/TabunKitchenInvestments-NDA.txt,1292,1347,"Confidential Information includes, without lim...","[{""file_path"": ""contractnli/TabunKitchenInvest..."
1,contractnli,Consider Grindrod SA's Non-Disclosure Agreemen...,1,contractnli/Grindrod%20SA%20Confidentiality%20...,757,1492,1.1 “Confidential Information” means; all tech...,"[{""file_path"": ""contractnli/Grindrod%20SA%20Co..."
2,contractnli,Consider the Data Use Agreement in New York Ci...,1,contractnli/Data Use Agreement New York City.txt,6985,7273,1. Only the Data Recipient’s employees and/or ...,"[{""file_path"": ""contractnli/Data Use Agreement..."
3,contractnli,Consider the Mutual Non-Disclosure Agreement b...,1,contractnli/wayne-fueling-systems-mutual-non-d...,968,1267,"Confidential Information also includes, but is...","[{""file_path"": ""contractnli/wayne-fueling-syst..."
4,contractnli,Consider FullStory's Mutual Non-Disclosure Agr...,1,contractnli/helpjuice_production%2Fuploads%2Fu...,3230,3396,Recipient agrees not to copy or reverse engine...,"[{""file_path"": ""contractnli/helpjuice_producti..."


In [5]:
df.groupby("dataset").size().rename("count")

dataset
contractnli    50
cuad           50
maud           50
privacy_qa     50
Name: count, dtype: int64

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save individual per-dataset JSON files (same format as the originals)
for dataset, tests in sampled_by_dataset.items():
    out_path = OUTPUT_DIR / f"{dataset}.json"
    with open(out_path, "w") as f:
        json.dump({"tests": tests}, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(tests):3d} tests → {out_path}")

# Save combined CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved combined CSV ({len(df)} rows) → {OUTPUT_CSV.resolve()}")

Saved  50 tests → ../data/LegalBenchRAG/benchmarks_50/contractnli.json
Saved  50 tests → ../data/LegalBenchRAG/benchmarks_50/cuad.json
Saved  50 tests → ../data/LegalBenchRAG/benchmarks_50/maud.json
Saved  50 tests → ../data/LegalBenchRAG/benchmarks_50/privacy_qa.json

Saved combined CSV (200 rows) → /home/twa174/LegalRAG/data/LegalBenchRAG/sampled_queries.csv
